
# CardioIA — Fase 2 · Parte 1: extração de sintomas e sugestão de diagnóstico

Este notebook demonstra o módulo [`src/extrator_sintomas.py`](../src/extrator_sintomas.py), que aplica **NLP baseado em regras** (abordagem simbólica, Cap. 10) para:

1. ler os relatos de [`data/frases_sintomas.txt`](../data/frases_sintomas.txt);
2. reconhecer sintomas usando o mapa de conhecimento [`data/mapa_conhecimento.csv`](../data/mapa_conhecimento.csv);
3. extrair **quando começou** e **como afeta a rotina** com expressões regulares (NER por regex);
4. sugerir um diagnóstico e as alternativas mais próximas.

> Material educacional. A sugestão não substitui avaliação médica.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import extrator_sintomas as ext

pd.set_option("display.max_colwidth", None)


## 1. Entradas: relatos e mapa de conhecimento

In [2]:
frases = ext.carregar_frases()
for i, f in enumerate(frases, start=1):
    print(f"{i:02d}. {f}")

01. Há dois dias estou com uma dor no peito que piora quando faço esforço físico e alivia quando descanso, por isso parei de subir as escadas do trabalho.
02. Desde ontem à noite sinto um aperto forte no peito que se espalha para o braço esquerdo, com suor frio e enjoo, e não consegui sair da cama hoje.
03. Sinto cansaço constante há uma semana, mesmo depois de descansar, minhas pernas ficam inchadas no fim do dia e preciso dormir com dois travesseiros para conseguir respirar, o que me impede de trabalhar.
04. Há três semanas meu coração dispara do nada e fica batendo irregular, com tontura, e tive que parar de dirigir por medo de passar mal.
05. Hoje de manhã acordei com uma dor de cabeça forte na nuca e a vista embaçada, medi a pressão e estava muito alta, então faltei ao trabalho.
06. Há quatro dias, depois de uma gripe, tenho uma dor no peito que piora quando respiro fundo ou deito e melhora quando me inclino para frente, e não consigo dormir direito.
07. Desde a viagem de avião de

In [3]:
mapa = pd.read_csv(ext.ARQ_MAPA)
print(f"{len(mapa)} linhas | {mapa['Doença Associada'].nunique()} doenças")
mapa.head(12)

64 linhas | 13 doenças


,Sintoma 1,Sintoma 2,Doença Associada
0,dor no peito,aperto no peito,Infarto Agudo do Miocárdio
1,aperto no tórax,pressão no peito,Infarto Agudo do Miocárdio
2,dor no braço esquerdo,espalha para o braço esquerdo,Infarto Agudo do Miocárdio
3,dor na mandíbula,dor que não passa,Infarto Agudo do Miocárdio
4,suor frio,sudorese,Infarto Agudo do Miocárdio
5,enjoo,náusea,Infarto Agudo do Miocárdio
6,dor no peito em repouso,aperto forte no peito,Infarto Agudo do Miocárdio
7,dor no peito,desconforto no peito,Angina Estável
8,dor no peito ao esforço,piora quando faço esforço,Angina Estável
9,dor ao subir escada,aperto no peito ao caminhar,Angina Estável


In [4]:
mapa["Doença Associada"].value_counts().rename("linhas no mapa").to_frame()

,linhas no mapa
Doença Associada,
Insuficiência Cardíaca,8
Infarto Agudo do Miocárdio,7
Arritmia Cardíaca,6
Trombose Venosa Profunda,6
Angina Estável,5
Crise Hipertensiva,5
Pericardite,5
Estenose Aórtica,5
Endocardite Infecciosa,5



## 2. Normalização (análise morfológica)

Cada expressão do mapa e cada frase passam pelo mesmo pipeline: minúsculas → tokens → remoção de *stopwords* (mantendo **não / nem / sem / nunca**, que carregam negação) → **estemização RSLP** (NLTK). Assim, "pernas inchadas" e "perna inchada" viram os mesmos radicais.

In [5]:
exemplos = ["pernas inchadas", "perna inchada", "piora quando faço esforço", "não sinto dor no peito"]
pd.DataFrame({"texto": exemplos, "radicais": [ext.normalizar(e) for e in exemplos]})

,texto,radicais
0,pernas inchadas,"[pern, inch]"
1,perna inchada,"[pern, inch]"
2,piora quando faço esforço,"[pior, fac, esforc]"
3,não sinto dor no peito,"[nao, sint, dor, peit]"



## 3. Regras de reconhecimento

* **Dicionário:** uma expressão do mapa é reconhecida quando seus radicais aparecem na mesma ordem, permitindo até `SALTO_MAXIMO` palavras entre eles ("dor no peito **que piora** ao esforço").
* **Negação:** se *não / nem / sem / nunca* aparece até `JANELA_NEGACAO` palavras antes do sintoma, na mesma oração, ele é registrado como **negado**.
* **Tempo e impacto:** expressões regulares capturam "há dois dias", "desde ontem à noite", "parei de…", "não consigo…".
* **Pontuação:** cada sintoma soma `1 / nº de doenças ligadas a ele`. Sintomas genéricos (como *falta de ar*) pesam menos que sintomas específicos (como *tossi com sangue*), uma ideia parecida com o IDF.

In [6]:
expressao_doencas, radicais = ext.carregar_mapa()
teste = "Desde a viagem de avião, minha panturrilha está inchada, não sinto dor no peito, mas mal consigo caminhar."
afirmados, negados = ext.extrair_sintomas(teste, radicais)
print("Afirmados:", sorted(afirmados))
print("Negados  :", sorted(negados))
print("Início   :", ext.extrair_tempo(teste))
print("Impacto  :", ext.extrair_impacto(teste))

Afirmados: ['mal consigo caminhar', 'panturrilha inchada', 'viagem de avião']
Negados  : ['dor no peito']
Início   : ['desde a viagem de avião']
Impacto  : ['mal consigo caminhar']



## 4. Diagnóstico sugerido para as 10 frases

In [7]:
resultados = ext.analisar_arquivo()
ext.imprimir_relatorio(resultados)


[01] Há dois dias estou com uma dor no peito que piora quando faço esforço físico e alivia quando descanso, por isso parei de subir as escadas do trabalho.
     Sintomas identificados : alivia quando descanso, dor no peito, dor no peito ao esforço, piora quando faço esforço
     Início                 : há dois dias
     Impacto na rotina      : parei de subir as escadas do trabalho
     Diagnóstico sugerido   : Angina Estável (confiança relativa 71%)
     Alternativas           : Estenose Aórtica (0.50), Infarto Agudo do Miocárdio (0.33)

[02] Desde ontem à noite sinto um aperto forte no peito que se espalha para o braço esquerdo, com suor frio e enjoo, e não consegui sair da cama hoje.
     Sintomas identificados : aperto forte no peito, aperto no peito, enjoo, espalha para o braço esquerdo, suor frio
     Início                 : desde ontem à noite, hoje
     Impacto na rotina      : não consegui sair da cama hoje
     Diagnóstico sugerido   : Infarto Agudo do Miocárdio (confiança

In [8]:
tabela = pd.DataFrame(resultados)[["frase", "sintomas", "inicio", "impacto_rotina", "diagnostico_sugerido", "confianca_relativa", "alternativas"]]
tabela.index = range(1, len(tabela) + 1)
tabela

,frase,sintomas,inicio,impacto_rotina,diagnostico_sugerido,confianca_relativa,alternativas
1,"Há dois dias estou com uma dor no peito que piora quando faço esforço físico e alivia quando descanso, por isso parei de subir as escadas do trabalho.","[alivia quando descanso, dor no peito, dor no peito ao esforço, piora quando faço esforço]",[há dois dias],[parei de subir as escadas do trabalho],Angina Estável,0.71,"[Estenose Aórtica (0.50), Infarto Agudo do Miocárdio (0.33)]"
2,"Desde ontem à noite sinto um aperto forte no peito que se espalha para o braço esquerdo, com suor frio e enjoo, e não consegui sair da cama hoje.","[aperto forte no peito, aperto no peito, enjoo, espalha para o braço esquerdo, suor frio]","[desde ontem à noite, hoje]",[não consegui sair da cama hoje],Infarto Agudo do Miocárdio,1.00,[]
3,"Sinto cansaço constante há uma semana, mesmo depois de descansar, minhas pernas ficam inchadas no fim do dia e preciso dormir com dois travesseiros para conseguir respirar, o que me impede de trabalhar.","[cansaço, cansaço constante, dormir com travesseiros, perna inchada, pernas inchadas]",[há uma semana],[me impede de trabalhar],Insuficiência Cardíaca,0.70,"[Trombose Venosa Profunda (1.00), Miocardite (0.50)]"
4,"Há três semanas meu coração dispara do nada e fica batendo irregular, com tontura, e tive que parar de dirigir por medo de passar mal.","[batendo irregular, batimento irregular, coração dispara, tontura]",[há três semanas],[tive que parar de dirigir por medo de passar mal],Arritmia Cardíaca,0.83,"[Crise Hipertensiva (0.33), Estenose Aórtica (0.33)]"
5,"Hoje de manhã acordei com uma dor de cabeça forte na nuca e a vista embaçada, medi a pressão e estava muito alta, então faltei ao trabalho.","[dor de cabeça, dor de cabeça forte, dor na nuca, pressão alta, pressão muito alta, vista embaçada]",[hoje de manhã],[faltei ao trabalho],Crise Hipertensiva,1.00,[]
6,"Há quatro dias, depois de uma gripe, tenho uma dor no peito que piora quando respiro fundo ou deito e melhora quando me inclino para frente, e não consigo dormir direito.","[depois de uma gripe, dor ao respirar, dor ao respirar fundo, dor no peito, dor no peito ao respirar, melhora inclinado para frente, melhora quando me inclino, piora quando deito, piora quando respiro fundo]",[há quatro dias],[não consigo dormir direito],Pericardite,0.65,"[Embolia Pulmonar (2.00), Miocardite (0.50)]"
7,"Desde a viagem de avião de dez horas, há cinco dias, minha panturrilha direita está inchada, vermelha e dolorida, não sinto dor no peito, mas mal consigo caminhar até a padaria.","[mal consigo caminhar, panturrilha inchada, viagem de avião]","[desde a viagem de avião, há cinco dias]",[mal consigo caminhar até a padaria],Trombose Venosa Profunda,1.00,[]
8,"Ontem desmaiei enquanto corria no parque e, há cerca de um mês, sinto tontura e falta de ar quando faço exercício, então parei de treinar.","[desmaiei, desmaiei enquanto corria, desmaio, falta de ar, falta de ar quando faço exercício, tontura]","[ontem, há cerca de um mês]",[parei de treinar],Estenose Aórtica,0.72,"[Crise Hipertensiva (0.33), Arritmia Cardíaca (0.33)]"
9,"Há duas semanas, depois de um tratamento dentário, tenho febre que vai e volta, calafrios e suor à noite, e me sinto tão fraco que não consigo cuidar das tarefas de casa.","[calafrios, febre, febre que vai e volta, sinto fraco, suor à noite, tratamento dentário]",[há duas semanas],[não consigo cuidar das tarefas de casa],Endocardite Infecciosa,0.92,[Miocardite (0.50)]
10,"De repente, há duas horas, senti falta de ar intensa e dor no peito ao respirar, tossi com sangue e não consigo falar frases inteiras sem parar para puxar o ar.","[dor ao respirar, dor no peito, dor no peito ao respirar, falta de ar, falta de ar intensa, tosse com sangue, tossi com sangue]","[de repente, há duas horas]",[não consigo falar frases inteiras sem parar para puxar o ar],Embolia Pulmonar,0.76,"[Angina Estável (0.67), Infarto Agudo do Miocárdio (0.33)]"


In [9]:
ext.salvar_resultados(resultados)
print("Resultado salvo em", ext.ARQ_SAIDA.relative_to(RAIZ).as_posix())

Resultado salvo em outputs/diagnosticos_parte1.csv



## 5. Mapa de conhecimento como ontologia (RDF)

Para aproximar o mapa de uma ontologia, como no Cap. 10 (rdflib), ele também é exportado em RDF/Turtle com as classes `Sintoma` e `Doenca` e a propriedade `indicaDoenca`. A consulta SPARQL abaixo responde: *quais sintomas indicam Insuficiência Cardíaca?*

In [10]:
g = ext.exportar_ontologia()
print(f"{len(g)} triplas salvas em {ext.ARQ_ONTOLOGIA.relative_to(RAIZ).as_posix()}")

consulta = '''
PREFIX cardio: <http://cardioia.example.org/ontologia#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?sintoma WHERE {
    ?s cardio:indicaDoenca ?d .
    ?d rdfs:label "Insuficiência Cardíaca"@pt .
    ?s rdfs:label ?sintoma .
} ORDER BY ?sintoma
'''
[str(linha.sintoma) for linha in g.query(consulta)]

385 triplas salvas em outputs/ontologia_cardioia.ttl

['acordo com falta de ar',
 'barriga inchada',
 'cansaço',
 'cansaço constante',
 'dificuldade para respirar',
 'dormir com travesseiros',
 'fadiga',
 'falta de ar',
 'falta de ar deitado',
 'falta de ar à noite',
 'fraqueza',
 'ganho de peso rápido',
 'inchaço nas pernas',
 'pernas inchadas',
 'pés inchados',
 'tornozelos inchados']


## 6. Limitações observadas

* **Cobertura do vocabulário:** o sistema só reconhece o que está no mapa. Gírias ou descrições novas ("meu peito está esquisito") passam despercebidas até alguém incluí-las.
* **Escopo da negação:** a regra é local. Na frase 10, "sem parar para **puxar o ar**" foi marcado como negado, embora o paciente *precise* parar para respirar. Isso mostra um caso de ambiguidade sintática que regras simples não resolvem.
* **Sobreposição entre doenças:** "dor no peito" e "falta de ar" aparecem em várias doenças. A ponderação reduz o peso desses sintomas, mas a decisão depende de existir pelo menos um sintoma específico.
* **Explicabilidade:** como vantagem, cada sugestão vem acompanhada das evidências usadas. Isso facilita a auditoria por um profissional de saúde, um requisito importante em sistemas de alto risco.